# 04 - Zero-shot comparison and consensus
**Cantaloupe flowers - zero-shot pseudo-labels (no ground truth)**

Compares the zero-shot detectors with label-free checks: mirror and copy consistency, agreement between models, box counts. Builds consensus pseudo-labels (boxes found by at least `CONSENSUS_MIN` models) and a hand-audit sheet. If a filled `audit.csv` is attached, it reports precision.

No training and no labels: nothing here is an accuracy unless the audit is filled.

Dataset: https://www.kaggle.com/datasets/samirhossain2001/cantaloupe-flower-datasets

**Attach as inputs:** the dataset + the outputs of 00 and of every zero-shot notebook (+ the mask notebook, + a filled `audit.csv`, both optional)

## 1 - Setup

In [ ]:
import os
import gc
import re
import json
import glob
import time
import random
import hashlib
from datetime import datetime
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import cv2
import torch
import torchvision
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

NB_START = time.time()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| cv2", cv2.__version__,
      "| device", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu", "| cpu cores", os.cpu_count())

## 2 - Configuration

In [ ]:
@dataclass
class CFG:
    INPUT_ROOTS: list = field(default_factory=lambda: os.environ.get("INPUT_ROOTS", '/kaggle/input').split(os.pathsep))
    OUT_DIR: str = '/kaggle/working' if os.path.isdir('/kaggle/working') else "outputs"
    DATA_DIR: str = '/kaggle/input/datasets/samirhossain2001/cantaloupe-flower-datasets/Cantaloupe flower datasets'  # dataset folder on Kaggle, checked first; otherwise the inputs are searched
    TAGS: list = field(default_factory=lambda: ['owlv2', 'gdino'])  # box notebooks compared
    MASK_TAG: str = 'gdino_sam2'
    MATCH_IOU: float = 0.5
    CONSENSUS_MIN: int = 2  # a consensus box needs this many models
    AUDIT_N: int = 40  # test photos in the hand-audit sheet
    N_BOOT: int = 1000
    SEED: int = 42


cfg = CFG()
SMOKE = False
cfg

## 3 - Shared helpers

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def find_artifact(fname, required=True):
    for root in cfg.INPUT_ROOTS:
        hits = sorted(glob.glob(os.path.join(root, "**", fname), recursive=True))
        if hits:
            return hits[0]
    if required:
        raise FileNotFoundError(f"{fname} not found: attach the output of the notebook that writes it as an input.")
    return None


def find_data_root(rel_path):
    if os.path.exists(os.path.join(cfg.DATA_DIR, rel_path)):
        return cfg.DATA_DIR
    for root in cfg.INPUT_ROOTS:
        for dirpath, _, _ in os.walk(root):
            if os.path.exists(os.path.join(dirpath, rel_path)):
                return dirpath
    raise FileNotFoundError(f"{rel_path} not found: attach the dataset as an input.")


def read_rgb(rel_path):
    a = cv2.imread(os.path.join(DATA_ROOT, rel_path), cv2.IMREAD_COLOR)
    assert a is not None, f"unreadable image: {rel_path}"
    return cv2.cvtColor(a, cv2.COLOR_BGR2RGB)


def dihedral(a, k):
    """k = 0..3: rotate k x 90 degrees counter-clockwise; k = 4..7: the same, then mirror left-right."""
    a = np.rot90(a, k % 4)
    return np.ascontiguousarray(a[:, ::-1] if k >= 4 else a)


def inv_k(k):
    """Inverse of dihedral(., k): rotations undo with the opposite turn, mirrored ones are their own inverse."""
    return k if k >= 4 else (4 - k) % 4


def dihedral_points(xy, k, w, h):
    xy = np.asarray(xy, dtype=np.float64).reshape(-1, 2).copy()
    for _ in range(k % 4):
        xy = np.stack([xy[:, 1], w - xy[:, 0]], axis=1)
        w, h = h, w
    if k >= 4:
        xy[:, 0] = w - xy[:, 0]
    return xy


def dihedral_boxes(boxes, k, w, h):
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    if len(boxes) == 0:
        return boxes
    p1, p2 = dihedral_points(boxes[:, :2], k, w, h), dihedral_points(boxes[:, 2:], k, w, h)
    return np.concatenate([np.minimum(p1, p2), np.maximum(p1, p2)], axis=1)


def box_area(b):
    return (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])


def box_iou(a, b):
    a, b = np.asarray(a, dtype=np.float64).reshape(-1, 4), np.asarray(b, dtype=np.float64).reshape(-1, 4)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(rb - lt, 0, None).prod(-1)
    return inter / np.maximum(box_area(a)[:, None] + box_area(b)[None, :] - inter, 1e-9)


def match_f1_from_iou(iou, n_a, n_b, thr):
    """F1 of a one-to-one matching at IoU >= thr; nan when both sides are empty."""
    if n_a + n_b == 0:
        return float("nan")
    if n_a == 0 or n_b == 0:
        return 0.0
    r, c = linear_sum_assignment(-iou)
    return 2 * int((iou[r, c] >= thr).sum()) / (n_a + n_b)


def match_f1(a, b, thr):
    return match_f1_from_iou(box_iou(a, b), len(a), len(b), thr)


def boot_ci(values, n_boot, seed=0):
    v = np.asarray([x for x in values if not np.isnan(x)], dtype=np.float64)
    if len(v) == 0:
        return float("nan"), float("nan"), float("nan")
    means = np.random.default_rng(seed).choice(v, (n_boot, len(v))).mean(1)
    return float(v.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def hex_rgb(h):
    return tuple(int(h[i:i + 2], 16) for i in (1, 3, 5))


def tag_rect(img, text, x, y, scale):
    """Rectangle of a text tag whose bottom-left corner is (x, y), kept inside the image."""
    (tw, th), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
    x = int(min(max(x, 0), img.shape[1] - tw - 6))
    y = int(min(max(y, th + base + 4), img.shape[0]))
    return x, y - th - base - 4, x + tw + 6, y, base


def put_tag(img, text, x, y, rgb, scale):
    x1, y1, x2, y2, base = tag_rect(img, text, x, y, scale)
    cv2.rectangle(img, (x1, y1), (x2, y2), rgb, -1)
    cv2.putText(img, text, (x1 + 3, y2 - base - 1), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 2, cv2.LINE_AA)
    return x1, y1, x2, y2


def draw_boxes(img, boxes, scores=None, color="#2a78d6", numbers=False, thick=None):
    """Thick box with a dark outline, so it shows on leaves and on bright sky; label on a filled tag at the first
    corner of the box where it does not cover an earlier tag."""
    out = img.copy()
    rgb = hex_rgb(color)
    side = max(out.shape[:2])
    t = thick or max(3, round(side / 160))
    scale = max(0.6, side / 900)
    placed = []
    for i, b in enumerate(np.asarray(boxes).reshape(-1, 4).astype(int)):
        p1, p2 = (int(b[0]), int(b[1])), (int(b[2]), int(b[3]))
        cv2.rectangle(out, p1, p2, (20, 20, 20), t + 3, cv2.LINE_AA)
        cv2.rectangle(out, p1, p2, rgb, t, cv2.LINE_AA)
        label = str(i) if numbers else (f"{scores[i]:.2f}" if scores is not None else "")
        if not label:
            continue
        (tw, th), base = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
        h = th + base + 4
        o = (t + 3) // 2
        corners = [(p1[0] - o, p1[1] - o), (p1[0] - o, p2[1] + o + h), (p2[0] - tw - 6 + o, p1[1] - o),
                   (p2[0] - tw - 6 + o, p2[1] + o + h), (p1[0] + o, p1[1] + o + h)]
        free = [c for c in corners if not any(
            r[0] < q[2] and q[0] < r[2] and r[1] < q[3] and q[1] < r[3] for r in [tag_rect(out, label, *c, scale)[:4]] for q in placed)]
        placed.append(put_tag(out, label, *(free[0] if free else corners[0]), rgb, scale))
    return out


def finish(fig, title):
    """Title above the panels (also for tall figures), then show."""
    h = fig.get_figheight()
    fig.suptitle(title, y=1 - 0.12 / h, va="top")
    plt.tight_layout(rect=(0, 0, 1, 1 - 0.5 / h))
    plt.show()


def save_json(obj, name):
    path = os.path.join(cfg.OUT_DIR, name)
    with open(path, "w") as f:
        json.dump(obj, f, indent=1)
    return path


os.makedirs(cfg.OUT_DIR, exist_ok=True)
seed_everything(cfg.SEED)

probe = np.zeros((40, 60), np.uint8)
probe[5:12, 30:50] = 1
for k in range(8):
    ys, xs = np.nonzero(dihedral(probe, k))
    assert np.allclose(dihedral_boxes([[30, 5, 50, 12]], k, 60, 40)[0], [xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]), k
    assert np.array_equal(dihedral(dihedral(probe, k), inv_k(k)), probe), k
print("box transform matches the image transform, and each inverse undoes it, for all 8 flips / rotations")

## 4 - Load every run
Every run must come from the same 00 split.

In [ ]:
split_meta = json.load(open(find_artifact("split.json")))
ITEMS = pd.DataFrame([dict(it, split=s) for s, its in split_meta["items"].items() for it in its])
DATA_ROOT = find_data_root(ITEMS.img.iloc[0])
M, DETS, MIR = {}, {}, {}
for tag in cfg.TAGS:
    M[tag] = json.load(open(find_artifact(f"metrics_{tag}.json")))
    assert M[tag]["split_id"] == split_meta["split_id"], f"{tag} ran on another 00 output: rerun it"
    pseudo = json.load(open(find_artifact(f"pseudo_{tag}.json")))
    assert pseudo["info"].get("propagated"), f"{tag} output is from an older version: rerun it with the current notebook"
    img_of = {im["id"]: im["item_id"] for im in pseudo["images"]}
    b, s = defaultdict(list), defaultdict(list)
    for a in pseudo["annotations"]:
        x, y, w, h = a["bbox"]
        b[img_of[a["image_id"]]].append([x, y, x + w, y + h])
        s[img_of[a["image_id"]]].append(a["score"])
    DETS[tag] = {i: {"boxes": np.array(b[i], dtype=np.float64).reshape(-1, 4), "scores": np.array(s[i])} for i in img_of.values()}
    MIR[tag] = {i: np.array(v, dtype=np.float64).reshape(-1, 4) for i, v in json.load(open(find_artifact(f"mirror_{tag}.json"))).items()}
common_ids = sorted(set.intersection(*[set(d) for d in DETS.values()]))
ITEMS = ITEMS[ITEMS.id.isin(common_ids)].reset_index(drop=True)
BY_ID = ITEMS.set_index("id", drop=False)
mirror_ids = sorted(set.intersection(*[set(d) for d in MIR.values()]))
mask_path = find_artifact(f"metrics_{cfg.MASK_TAG}.json", required=False)
MASK_M = json.load(open(mask_path)) if mask_path else None
COLORS = {t: PALETTE[k] for k, t in enumerate(cfg.TAGS + ["consensus"])}
print(f"split {split_meta['split_id']} | models {cfg.TAGS} | images scored by every model: {len(common_ids)} | "
      f"mirrored photos: {len(mirror_ids)} | mask notebook: {'found' if MASK_M else 'not attached'}")

## 5 - Summary

In [ ]:
rows = []
for t in cfg.TAGS:
    m = M[t]
    rows.append({"model": m["model"], "prompt": m["prompt"], "threshold": m["threshold"],
                 "val selection": m["validation"]["selection_score"],
                 "test mirror F1": m["test"]["mirror_f1"], "95% CI": "[{:.3f}, {:.3f}]".format(*m["test"]["mirror_ci"]),
                 "copies colour-gray F1": m["test"]["copy_by_kind"].get("colour-gray", {}).get("f1"),
                 "boxes / photo": m["test"]["det_per_ref"], "photos with a box": m["test"]["share_ref_with_det"],
                 "params (M)": m["params_m"], "latency (ms)": m["latency_ms"]})
if MASK_M:
    rows.append({"model": MASK_M["model"] + " (masks from " + MASK_M["boxes_from"] + " boxes)",
                 "test mirror F1": MASK_M["test"]["mask_mirror_iou"],
                 "95% CI": "[{:.3f}, {:.3f}]".format(*MASK_M["test"]["mirror_ci"]), "params (M)": MASK_M["params_m"]})
SUMMARY = pd.DataFrame(rows)
print("For the mask row the value is the mean mask IoU between the original and the mirrored pass, not an F1.")
SUMMARY.round(3)

## 6 - Copy consistency and box counts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
kinds = ["mirror (colour)", "colour-gray", "gray-gray"]
w = 0.8 / len(cfg.TAGS)
for k, t in enumerate(cfg.TAGS):
    te = M[t]["test"]
    src = {"mirror (colour)": {"f1": te["mirror_f1"], "ci": te["mirror_ci"]}, **te["copy_by_kind"]}
    vals = [src.get(x, {}).get("f1", np.nan) for x in kinds]
    err = np.array([[v - src[x]["ci"][0], src[x]["ci"][1] - v] if x in src else [0, 0] for x, v in zip(kinds, vals)]).T
    axes[0].bar(np.arange(len(kinds)) + k * w, vals, w, yerr=np.abs(err), capsize=3, color=COLORS[t], label=M[t]["model"])
axes[0].set_xticks(np.arange(len(kinds)) + w * (len(cfg.TAGS) - 1) / 2)
axes[0].set_xticklabels(kinds)
axes[0].set_ylim(0, 1)
axes[0].set_title("Test consistency F1 (95% CI): mirror pass and Roboflow copies", fontsize=10)
axes[0].legend(fontsize=8)
test_refs = list(ITEMS[(ITEMS.split == "test") & ITEMS.is_ref].id)
for t in cfg.TAGS:
    axes[1].hist([len(DETS[t][i]["boxes"]) for i in test_refs], bins=range(0, 16), alpha=0.55, color=COLORS[t], label=M[t]["model"])
axes[1].set_title("Boxes per test photo", fontsize=10)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7 - Agreement between models

In [ ]:
def agreement(ids, a, b):
    return np.array([match_f1(DETS[a][i]["boxes"], DETS[b][i]["boxes"], cfg.MATCH_IOU) for i in ids])


n = len(cfg.TAGS)
A = np.eye(n)
for x in range(n):
    for y in range(x + 1, n):
        v = agreement(test_refs, cfg.TAGS[x], cfg.TAGS[y])
        A[x, y] = A[y, x] = np.nanmean(v) if np.isfinite(v).any() else np.nan
        mean, lo, hi = boot_ci(v, cfg.N_BOOT, cfg.SEED)
        print(f"{cfg.TAGS[x]} vs {cfg.TAGS[y]}: box agreement F1 {mean:.3f} [95% CI {lo:.3f}, {hi:.3f}] over {int(np.isfinite(v).sum())} test photos")
fig, ax = plt.subplots(figsize=(1.8 * n + 2, 1.6 * n + 1.5))
im = ax.imshow(A, vmin=0, vmax=1, cmap="Blues")
for x in range(n):
    for y in range(n):
        ax.text(y, x, f"{A[x, y]:.2f}", ha="center", va="center", color="white" if A[x, y] > 0.6 else "black")
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels([M[t]["model"] for t in cfg.TAGS], rotation=20, fontsize=8)
ax.set_yticklabels([M[t]["model"] for t in cfg.TAGS], fontsize=8)
ax.set_title("Box agreement between models (test photos, F1 at IoU >= MATCH_IOU)", fontsize=10)
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 8 - Consensus pseudo-labels
Boxes are pooled by score; each keeps the best-overlapping box of every other model; clusters with enough models become one score-weighted box.

In [ ]:
def consensus(per_model):
    """per_model: list of (boxes, scores), one per model. Returns score-weighted boxes found by >= CONSENSUS_MIN models."""
    pool = [(float(s), k, b) for k, (bx, sc) in enumerate(per_model) for b, s in zip(bx, sc)]
    pool.sort(key=lambda p: -p[0])
    used, out_b, out_s = set(), [], []
    for a, (s, k, b) in enumerate(pool):
        if a in used:
            continue
        members = [(s, b)]
        used.add(a)
        for other in range(len(per_model)):
            if other == k:
                continue
            cand = [(box_iou([b], [pb])[0, 0], c) for c, (ps, pk, pb) in enumerate(pool) if pk == other and c not in used]
            cand = [x for x in cand if x[0] >= cfg.MATCH_IOU]
            if cand:
                c = max(cand)[1]
                used.add(c)
                members.append((pool[c][0], pool[c][2]))
        if len(members) >= cfg.CONSENSUS_MIN:
            wts = np.array([m[0] for m in members])
            out_b.append((np.stack([m[1] for m in members]) * wts[:, None]).sum(0) / wts.sum())
            out_s.append(float(wts.mean()))
    return {"boxes": np.array(out_b, dtype=np.float64).reshape(-1, 4), "scores": np.array(out_s)}


DETS["consensus"] = {i: consensus([(DETS[t][i]["boxes"], DETS[t][i]["scores"]) for t in cfg.TAGS]) for i in tqdm(list(ITEMS.id), desc="consensus")}
MIR["consensus"] = {i: consensus([(MIR[t][i], np.ones(len(MIR[t][i]))) for t in cfg.TAGS])["boxes"] for i in mirror_ids}
cons_rows = {}
for s in ("validation", "test"):
    refs = [i for i in ITEMS[(ITEMS.split == s) & ITEMS.is_ref].id if i in MIR["consensus"]]
    f = [match_f1(DETS["consensus"][i]["boxes"], MIR["consensus"][i], cfg.MATCH_IOU) for i in refs]
    mean, lo, hi = boot_ci(f, cfg.N_BOOT, cfg.SEED)
    n = [len(DETS["consensus"][i]["boxes"]) for i in refs]
    cons_rows[s] = {"mirror_f1": mean, "mirror_ci": [lo, hi], "det_per_ref": float(np.mean(n)) if n else 0.0,
                    "share_ref_with_det": float(np.mean([x > 0 for x in n])) if n else 0.0}
    print(f"consensus ({cfg.CONSENSUS_MIN}+ models agree) {s:10s}: mirror consistency F1 {mean:.3f} [95% CI {lo:.3f}, {hi:.3f}] | "
          f"boxes per photo {cons_rows[s]['det_per_ref']:.2f} | photos with a box {cons_rows[s]['share_ref_with_det']:.0%}")

## 9 - Where the models disagree

In [ ]:
agree_mean = pd.Series({i: np.nanmean([match_f1(DETS[cfg.TAGS[x]][i]["boxes"], DETS[cfg.TAGS[y]][i]["boxes"], cfg.MATCH_IOU)
                                         for x in range(len(cfg.TAGS)) for y in range(x + 1, len(cfg.TAGS))]) for i in test_refs}).dropna()
worst = agree_mean.sort_values().index[:6].tolist()
if worst:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10), squeeze=False)
    for a, i in zip(axes.ravel(), worst):
        img = read_rgb(BY_ID.img[i])
        for k, t in enumerate(cfg.TAGS):
            img = draw_boxes(img, DETS[t][i]["boxes"], color=COLORS[t], thick=4 + 4 * (len(cfg.TAGS) - 1 - k))
        a.imshow(img)
        a.set_title(f"{BY_ID.source[i]} | agreement {agree_mean[i]:.2f}", fontsize=9)
    for a in axes.ravel():
        a.axis("off")
    fig.legend(handles=[plt.Rectangle((0, 0), 1, 1, fc="none", ec=COLORS[t], lw=3, label=M[t]["model"]) for t in cfg.TAGS],
               loc="lower center", ncol=len(cfg.TAGS), fontsize=11, frameon=False)
    plt.suptitle("Test photos where the models disagree most (earlier models drawn thicker, underneath)")
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    plt.show()

## 10 - Hand-audit sheet
The only route to a real precision number. Recall still needs full labels.

In [ ]:
rng = np.random.default_rng(cfg.SEED)
AUDIT_IDS = sorted(rng.choice(test_refs, min(cfg.AUDIT_N, len(test_refs)), replace=False).tolist())
audit_dir = os.path.join(cfg.OUT_DIR, "audit")
os.makedirs(audit_dir, exist_ok=True)
rows = []
for n_img, i in enumerate(AUDIT_IDS):
    img = read_rgb(BY_ID.img[i])
    panels = []
    for t in cfg.TAGS + ["consensus"]:
        d = DETS[t][i]
        panels.append(draw_boxes(img, d["boxes"], color=COLORS[t], numbers=True))
        put_tag(panels[-1], t, 6, 40, hex_rgb(COLORS[t]), 1.0)
        for k, (b, s) in enumerate(zip(d["boxes"], d["scores"])):
            rows.append({"audit_image": f"audit_{n_img:03d}.jpg", "item_id": i, "model": t, "box": k,
                         "x1": round(b[0], 1), "y1": round(b[1], 1), "x2": round(b[2], 1), "y2": round(b[3], 1),
                         "score": round(float(s), 4), "correct": ""})
    cv2.imwrite(os.path.join(audit_dir, f"audit_{n_img:03d}.jpg"), cv2.cvtColor(np.concatenate(panels, axis=1), cv2.COLOR_RGB2BGR))
pd.DataFrame(rows).to_csv(os.path.join(cfg.OUT_DIR, "audit_sheet.csv"), index=False)
print(f"audit sheet: {len(rows)} boxes on {len(AUDIT_IDS)} test photos -> audit_sheet.csv + audit/*.jpg")

filled = find_artifact("audit.csv", required=False)
if filled:
    au = pd.read_csv(filled)
    au = au[au.correct.isin([0, 1, "0", "1"])].astype({"correct": int})
    for t, g in au.groupby("model"):
        mean, lo, hi = boot_ci(g.correct.values.astype(float), cfg.N_BOOT, cfg.SEED)
        print(f"hand-audited precision {t:10s}: {mean:.3f} [95% CI {lo:.3f}, {hi:.3f}] over {len(g)} boxes")
else:
    print("No filled audit.csv attached: precision is unknown. Fill the `correct` column (1 / 0) of audit_sheet.csv, "
          "save it as audit.csv, attach it and rerun to get precision per model.")

## 11 - Saved outputs and verdict

In [ ]:
images, anns = [], []
for k, it in enumerate(ITEMS.sort_values("id").itertuples(), 1):
    images.append({"id": k, "file_name": it.img, "item_id": it.id, "split": it.split, "width": int(it.width), "height": int(it.height)})
    d = os.path.join(cfg.OUT_DIR, "yolo_consensus", "labels", it.split)
    os.makedirs(d, exist_ok=True)
    c = DETS["consensus"][it.id]
    with open(os.path.join(d, os.path.splitext(os.path.basename(it.img))[0] + ".txt"), "w") as f:
        for (x1, y1, x2, y2), s in zip(c["boxes"], c["scores"]):
            anns.append({"id": len(anns) + 1, "image_id": k, "category_id": 1, "bbox": [x1, y1, x2 - x1, y2 - y1],
                         "area": (x2 - x1) * (y2 - y1), "score": s, "iscrowd": 0})
            f.write(f"0 {(x1 + x2) / 2 / it.width:.6f} {(y1 + y2) / 2 / it.height:.6f} {(x2 - x1) / it.width:.6f} {(y2 - y1) / it.height:.6f}\n")
with open(os.path.join(cfg.OUT_DIR, "yolo_consensus", "data.yaml"), "w") as f:
    f.write("# labels only; images are in the dataset. Consensus pseudo-labels, not ground truth.\nnames:\n  0: flower\n")
save_json({"info": {"tag": "consensus", "models": cfg.TAGS, "min_models": cfg.CONSENSUS_MIN, "split_id": split_meta["split_id"],
                    "note": "zero-shot consensus pseudo-labels, not ground truth"},
           "images": images, "annotations": anns, "categories": [{"id": 1, "name": "flower"}]}, "pseudo_consensus.json")
SUMMARY.to_csv(os.path.join(cfg.OUT_DIR, "comparison_summary.csv"), index=False)
save_json({"split_id": split_meta["split_id"], "consensus": cons_rows, "agreement_matrix": A.tolist(), "tags": cfg.TAGS},
          "metrics_comparison.json")
ranked = sorted(cfg.TAGS, key=lambda t: -M[t]["test"]["mirror_f1"])
best = ranked[0]
tied = [t for t in ranked[1:] if M[t]["test"]["mirror_ci"][1] >= M[best]["test"]["mirror_ci"][0]]
print(f"Highest test mirror consistency: {M[best]['model']} ({M[best]['test']['mirror_f1']:.3f})"
      + (f"; its 95% CI overlaps {', '.join(M[t]['model'] for t in tied)}, so treat them as tied. " if tied else ". "))
print("Consistency and agreement are label-free checks; without the hand audit they say nothing about how many "
      "boxes are real flowers.")
print(f"saved pseudo_consensus.json, yolo_consensus/, comparison_summary.csv, metrics_comparison.json, audit/ | "
      f"total {(time.time() - NB_START) / 60:.1f} min")